In [0]:
# Bronze Layer Configuration
# Metadata-driven approach for loading tables from raw to bronze

from pyspark.sql.functions import col, lit, current_timestamp, md5, concat_ws, when, max as spark_max
from pyspark.sql.types import TimestampType
from datetime import datetime

# Table configuration with primary keys and load strategies
table_config = [
    {
        "table_name": "customers",
        "primary_keys": ["customer_id"],
        "load_strategy": "SCD2",  # Slowly Changing Dimension Type 2
        "source_schema": "retail.raw",
        "target_schema": "retail.bronze"
    },
    {
        "table_name": "addresses",
        "primary_keys": ["customer_id", "address_type"],
        "load_strategy": "SCD2",  # Slowly Changing Dimension Type 2
        "source_schema": "retail.raw",
        "target_schema": "retail.bronze"
    },
    {
        "table_name": "orders",
        "primary_keys": ["order_id"],
        "load_strategy": "APPEND",  # Append-only for fact table
        "source_schema": "retail.raw",
        "target_schema": "retail.bronze"
    },
    {
        "table_name": "payments",
        "primary_keys": ["payment_id"],
        "load_strategy": "APPEND",  # Append-only for fact table
        "source_schema": "retail.raw",
        "target_schema": "retail.bronze"
    },
    {
        "table_name": "refunds",
        "primary_keys": ["refund_id"],
        "load_strategy": "APPEND",  # Append-only for fact table
        "source_schema": "retail.raw",
        "target_schema": "retail.bronze"
    }
]

print("Configuration loaded for {} tables".format(len(table_config)))
for config in table_config:
    print(f"  - {config['table_name']}: {config['load_strategy']} (PK: {', '.join(config['primary_keys'])})")

In [0]:
# Helper function for SCD2 load strategy
def load_scd2(source_table, target_table, primary_keys, source_schema, target_schema):
    """
    Implements Slowly Changing Dimension Type 2 strategy.
    Tracks historical changes with start_date, end_date, and is_current flag.
    """
    try:
        source_df = spark.table(f"{source_schema}.{source_table}")
        
        # Check if target table exists
        target_exists = spark.catalog.tableExists(f"{target_schema}.{target_table}")
        
        if not target_exists:
            # Initial load: Add SCD2 columns
            bronze_df = source_df \
                .withColumn("created_timestamp", current_timestamp()) \
                .withColumn("updated_timestamp", current_timestamp()) \
                .withColumn("start_date", current_timestamp()) \
                .withColumn("end_date", lit(None).cast(TimestampType())) \
                .withColumn("is_current", lit(True))
            
            bronze_df.write.mode("overwrite").saveAsTable(f"{target_schema}.{target_table}")
            return {"status": "success", "operation": "initial_load", "records": bronze_df.count()}
        
        else:
            # Incremental load: Perform SCD2 merge
            target_df = spark.table(f"{target_schema}.{target_table}").filter(col("is_current") == True)
            
            # Create hash columns for comparison
            source_cols = [c for c in source_df.columns]
            hash_expr = concat_ws("", *[col(c).cast("string") for c in source_cols])
            
            source_with_hash = source_df.withColumn("source_hash", md5(hash_expr))
            target_with_hash = target_df.withColumn("target_hash", md5(concat_ws("|", *[col(c).cast("string") for c in source_cols])))
            
            # Join on primary keys
            join_condition = " AND ".join([f"source.{pk} = target.{pk}" for pk in primary_keys])
            
            # Find changed records
            changed_records = source_with_hash.alias("source") \
                .join(target_with_hash.alias("target"), join_condition, "inner") \
                .where("source.source_hash != target.target_hash") \
                .select("source.*")
            
            # Find new records
            new_records = source_with_hash.alias("source") \
                .join(target_with_hash.alias("target"), join_condition, "left_anti")
            
            # Expire old records
            if changed_records.count() > 0:
                expired_keys = changed_records.select(*primary_keys).distinct()
                
                # Update existing records to set is_current = False and end_date
                full_target_df = spark.table(f"{target_schema}.{target_table}")
                
                # Create condition for matching primary keys
                pk_conditions = [col(f"target.{pk}").isin([row[pk] for row in expired_keys.collect()]) for pk in primary_keys]
                combined_condition = pk_conditions[0]
                for condition in pk_conditions[1:]:
                    combined_condition = combined_condition & condition
                
                updated_target = full_target_df.alias("target") \
                    .withColumn("is_current", 
                               when(combined_condition & (col("is_current") == True), False)
                               .otherwise(col("is_current"))) \
                    .withColumn("end_date",
                               when(combined_condition & (col("is_current") == False) & col("end_date").isNull(), 
                                    current_timestamp())
                               .otherwise(col("end_date")))
                
                # Write updated target back
                updated_target.write.mode("overwrite").saveAsTable(f"{target_schema}.{target_table}")
            
            # Insert changed records as new versions
            if changed_records.count() > 0:
                changed_bronze = changed_records.drop("source_hash") \
                    .withColumn("created_timestamp", current_timestamp()) \
                    .withColumn("updated_timestamp", current_timestamp()) \
                    .withColumn("start_date", current_timestamp()) \
                    .withColumn("end_date", lit(None).cast(TimestampType())) \
                    .withColumn("is_current", lit(True))
                
                changed_bronze.write.mode("append").saveAsTable(f"{target_schema}.{target_table}")
            
            # Insert new records
            new_count = 0
            if new_records.count() > 0:
                new_bronze = new_records.drop("source_hash") \
                    .withColumn("created_timestamp", current_timestamp()) \
                    .withColumn("updated_timestamp", current_timestamp()) \
                    .withColumn("start_date", current_timestamp()) \
                    .withColumn("end_date", lit(None).cast(TimestampType())) \
                    .withColumn("is_current", lit(True))
                
                new_bronze.write.mode("append").saveAsTable(f"{target_schema}.{target_table}")
                new_count = new_records.count()
            
            return {
                "status": "success", 
                "operation": "incremental_load", 
                "new_records": new_count,
                "changed_records": changed_records.count()
            }
    
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# Helper function for APPEND load strategy
def load_append(source_table, target_table, primary_keys, source_schema, target_schema):
    """
    Implements append-only strategy for fact tables.
    Appends new records based on primary key.
    """
    try:
        source_df = spark.table(f"{source_schema}.{source_table}")
        
        # Check if target table exists
        target_exists = spark.catalog.tableExists(f"{target_schema}.{target_table}")
        
        if not target_exists:
            # Initial load
            bronze_df = source_df \
                .withColumn("created_timestamp", current_timestamp()) \
                .withColumn("updated_timestamp", current_timestamp())
            
            bronze_df.write.mode("overwrite").saveAsTable(f"{target_schema}.{target_table}")
            return {"status": "success", "operation": "initial_load", "records": bronze_df.count()}
        
        else:
            # Incremental load: Only append new records
            target_df = spark.table(f"{target_schema}.{target_table}")
            
            # Find new records based on primary key
            join_condition = " AND ".join([f"source.{pk} = target.{pk}" for pk in primary_keys])
            
            new_records = source_df.alias("source") \
                .join(target_df.alias("target").select(*primary_keys), join_condition, "left_anti")
            
            new_count = new_records.count()
            
            if new_count > 0:
                bronze_df = new_records \
                    .withColumn("created_timestamp", current_timestamp()) \
                    .withColumn("updated_timestamp", current_timestamp())
                
                bronze_df.write.mode("append").saveAsTable(f"{target_schema}.{target_table}")
            
            return {
                "status": "success", 
                "operation": "incremental_load", 
                "new_records": new_count
            }
    
    except Exception as e:
        return {"status": "failed", "error": str(e)}

print("Helper functions loaded successfully")
print("  - load_scd2(): For dimension tables")
print("  - load_append(): For fact tables")

In [0]:
# Main Bronze Load Processing
# Process all tables based on configuration

import time
from datetime import datetime

print("="*80)
print(f"Bronze Layer Load Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

results = []

for config in table_config:
    table_name = config['table_name']
    primary_keys = config['primary_keys']
    load_strategy = config['load_strategy']
    source_schema = config['source_schema']
    target_schema = config['target_schema']
    
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Processing: {table_name}")
    print(f"  Strategy: {load_strategy} | Primary Keys: {', '.join(primary_keys)}")
    
    start_time = time.time()
    
    try:
        # Route to appropriate load function based on strategy
        if load_strategy == "SCD2":
            result = load_scd2(
                source_table=table_name,
                target_table=table_name,
                primary_keys=primary_keys,
                source_schema=source_schema,
                target_schema=target_schema
            )
        elif load_strategy == "APPEND":
            result = load_append(
                source_table=table_name,
                target_table=table_name,
                primary_keys=primary_keys,
                source_schema=source_schema,
                target_schema=target_schema
            )
        else:
            result = {"status": "failed", "error": f"Unknown load strategy: {load_strategy}"}
        
        # Calculate execution time
        execution_time = round(time.time() - start_time, 2)
        result['table'] = table_name
        result['execution_time_sec'] = execution_time
        
        # Print result
        if result['status'] == 'success':
            if result['operation'] == 'initial_load':
                print(f"  ✓ Success: Initial load completed - {result.get('records', 0)} records")
            else:
                if load_strategy == "SCD2":
                    print(f"  ✓ Success: {result.get('new_records', 0)} new, {result.get('changed_records', 0)} changed")
                else:
                    print(f"  ✓ Success: {result.get('new_records', 0)} new records appended")
            print(f"  Execution time: {execution_time}s")
        else:
            print(f"  ✗ Failed: {result.get('error', 'Unknown error')}")
        
        results.append(result)
        
    except Exception as e:
        execution_time = round(time.time() - start_time, 2)
        error_result = {
            'table': table_name,
            'status': 'failed',
            'error': str(e),
            'execution_time_sec': execution_time
        }
        results.append(error_result)
        print(f"  ✗ Failed: {str(e)}")
        print(f"  Execution time: {execution_time}s")

print("\n" + "="*80)
print(f"Bronze Layer Load Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# Summary
successful = sum(1 for r in results if r['status'] == 'success')
failed = sum(1 for r in results if r['status'] == 'failed')
total_time = sum(r['execution_time_sec'] for r in results)

print(f"\nSummary:")
print(f"  Total tables: {len(results)}")
print(f"  Successful: {successful}")
print(f"  Failed: {failed}")
print(f"  Total execution time: {round(total_time, 2)}s")

# Display results as table
from pyspark.sql import Row
results_df = spark.createDataFrame([Row(**r) for r in results])
display(results_df)

In [0]:
%sql
TABLE retail.bronze.customers LIMIT 10;

In [0]:
%sql
TABLE retail.bronze.addresses LIMIT 10;


In [0]:
%sql
TABLE retail.bronze.orders LIMIT 10;


In [0]:
%sql
TABLE retail.bronze.payments LIMIT 10;


In [0]:
%sql
TABLE retail.bronze.refunds LIMIT 10;
